In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path='/Users/toszmac/Documents/Pharma_Manufacturing_Data_Analysis_Project/Pharma_Manufacturing_Data_Analysis_Folder/.env')

engine = create_engine(
    f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

print("Connected to lli_db")

Connected to lli_db


# Module 3 — Machine Utilization Analysis
## Batch Count per Machine by Stage (2025)

In [2]:
machine_wet_gran_query = """
    SELECT
        m.machine_name,
        m.machine_type,
        COUNT(f.fact_id) AS batch_count
    FROM fact_batch_production f
    JOIN dim_machine m ON f.machine_id = m.machine_id
    WHERE f.machine_id IS NOT NULL
    AND f.stage = 'wet_granulation'
    GROUP BY m.machine_name, m.machine_type
    ORDER BY batch_count DESC;
"""

df_wet_gran = pd.read_sql_query(machine_wet_gran_query, engine)
df_wet_gran

,machine_name,machine_type,batch_count
0,G1,granulator,451
1,G2,granulator,317


In [3]:
fig = px.bar(
    df_wet_gran,
    x='batch_count',
    y='machine_name',
    orientation='h',
    title='Machine Utilization — Wet Granulation (2025)',
    labels={'batch_count': 'Batch Count', 'machine_name': ''},
    text='batch_count',
    color='batch_count',
    color_continuous_scale='Blues'
)
fig.update_traces(textposition='auto', insidetextanchor='end')
fig.update_layout(
    plot_bgcolor='white',
    coloraxis_showscale=False,
    yaxis={'categoryorder': 'total ascending'},
    xaxis=dict(showgrid=False, title='')
)
fig.show()

In [4]:
machine_wet_gran_monthly_query = """
    SELECT
        TO_CHAR(d.full_date, 'YYYY-MM') AS month,
        TO_CHAR(d.full_date, 'Mon') AS month_short,
        m.machine_name,
        COUNT(f.fact_id) AS batch_count
    FROM fact_batch_production f
    JOIN dim_machine m ON f.machine_id = m.machine_id
    JOIN dim_date d ON f.date_id = d.date_id
    WHERE f.stage = 'wet_granulation'
    AND f.machine_id IS NOT NULL
    AND d.year = 2025
    GROUP BY
        TO_CHAR(d.full_date, 'YYYY-MM'),
        TO_CHAR(d.full_date, 'Mon'),
        m.machine_name
    ORDER BY month, m.machine_name;
"""

df_wet_gran_monthly = pd.read_sql_query(machine_wet_gran_monthly_query, engine)
df_wet_gran_monthly

,month,month_short,machine_name,batch_count
0,2025-01,Jan,G1,40
1,2025-01,Jan,G2,25
2,2025-02,Feb,G1,53
3,2025-02,Feb,G2,30
4,2025-03,Mar,G1,29
5,2025-03,Mar,G2,21
6,2025-04,Apr,G1,48
7,2025-04,Apr,G2,22
8,2025-05,May,G1,51
9,2025-05,May,G2,23


In [5]:
month_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

df_wet_gran_monthly['month_short'] = pd.Categorical(
    df_wet_gran_monthly['month_short'],
    categories=month_order,
    ordered=True
)
df_wet_gran_monthly = df_wet_gran_monthly.sort_values('month_short')

fig = px.bar(
    df_wet_gran_monthly,
    x='month_short',
    y='batch_count',
    color='machine_name',
    barmode='group',
    title='Monthly Batch Count by Granulator — Wet Granulation (2025)',
    labels={
        'batch_count': 'Batch Count',
        'month_short': '',
        'machine_name': 'Granulator'
    },
    text='batch_count',
    color_discrete_map={
        'G1': '#003f7f',
        'G2': '#7eb5d6'
    }
)

fig.update_traces(textposition='outside')
fig.update_layout(
    plot_bgcolor='white',
    xaxis=dict(showgrid=False),
    yaxis=dict(showgrid=False, title='Batch Count'),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)
fig.show()

## Wet Granulation
 
**Observation:**
G1 handled 451 batches while G2 handled 317, totaling 768 wet granulation batches out of 1,064 — meaning 296 batches skipped wet granulation entirely (dry blend products).
 
**Insight:**
G1 carries a significantly heavier load than G2, at roughly a 59/41 split. This asymmetry suggests either G1 has higher capacity, G2 has more downtime, or product scheduling consistently favors G1.
 
**Recommendation:**
Investigate whether G2 is being underutilized due to scheduling preference or actual capacity constraints. Rebalancing granulator assignments could reduce G1 wear and provide scheduling flexibility during peak months like Q3.


In [12]:
machine_dry_blend_query = """
    SELECT
        m.machine_name,
        m.machine_type,
        COUNT(f.fact_id) AS batch_count
    FROM fact_batch_production f
    JOIN dim_machine m ON f.machine_id = m.machine_id
    WHERE f.machine_id IS NOT NULL
    AND f.stage = 'dry_blending'
    GROUP BY m.machine_name, m.machine_type
    ORDER BY batch_count DESC;
"""

df_dry_blend = pd.read_sql_query(machine_dry_blend_query, engine)
df_dry_blend

,machine_name,machine_type,batch_count
0,B1 500L,blender,175
1,B2 500L,blender,160
2,B2 3000L,blender,157
3,B1 150L,blender,129
4,B2 1000L,blender,123
5,B2 150L,blender,120
6,B1 3000L,blender,107
7,B1 1000L,blender,93


In [13]:
fig = px.bar(
    df_dry_blend,
    x='batch_count',
    y='machine_name',
    orientation='h',
    title='Machine Utilization — Dry Blending (2025)',
    labels={'batch_count': 'Batch Count', 'machine_name': ''},
    text='batch_count',
    color='batch_count',
    color_continuous_scale='Blues'
)
fig.update_traces(textposition='auto', insidetextanchor='end')
fig.update_layout(
    plot_bgcolor='white',
    coloraxis_showscale=False,
    yaxis={'categoryorder': 'total ascending'},
    xaxis=dict(showgrid=False, title='')
)
fig.show()

## Dry Blending
 
**Observation:**
B1 500L leads at 175 batches, followed closely by B2 500L (160) and B2 3000L (157). B1 1000L trails significantly at 93 batches.
 
**Insight:**
The 500L blenders are the workhorses of the blending operation, which reflects that most products have smaller batch sizes suited to smaller blender volumes. The large-volume blenders (B1 3000L, B2 3000L) are used selectively, likely for high-volume generics like Butamirate Citrate.
 
**Recommendation:**
Monitor B1 1000L utilization — at only 93 batches it may be underutilized relative to its capacity. Cross-reference with batch sizes to determine if mid-volume batches are being routed to smaller blenders inefficiently.


In [14]:
machine_compression_query = """
    SELECT
        m.machine_name,
        m.machine_type,
        COUNT(f.fact_id) AS batch_count
    FROM fact_batch_production f
    JOIN dim_machine m ON f.machine_id = m.machine_id
    WHERE f.machine_id IS NOT NULL
    AND f.stage = 'compression'
    GROUP BY m.machine_name, m.machine_type
    ORDER BY batch_count DESC;
"""

df_compression = pd.read_sql_query(machine_compression_query, engine)
df_compression

,machine_name,machine_type,batch_count
0,ZP-41,tabletting,366
1,ACCURA D,tabletting,293
2,ACCURA B,tabletting,178
3,FLUIDPACK D-27,tabletting,15
4,KARNAVATI,tabletting,2


In [15]:
fig = px.bar(
    df_compression,
    x='batch_count',
    y='machine_name',
    orientation='h',
    title='Machine Utilization — Compression (2025)',
    labels={'batch_count': 'Batch Count', 'machine_name': ''},
    text='batch_count',
    color='batch_count',
    color_continuous_scale='Blues'
)
fig.update_traces(textposition='auto', insidetextanchor='end')
fig.update_layout(
    plot_bgcolor='white',
    coloraxis_showscale=False,
    yaxis={'categoryorder': 'total ascending'},
    xaxis=dict(showgrid=False, title='')
)
fig.show()

## Compression
 
**Observation:**
ZP-41 dominates at 366 batches, followed by ACCURA D at 293 and ACCURA B at 178. FLUIDPACK D-27 (15) and KARNAVATI (2) are barely used.
 
**Insight:**
Three machines handle virtually all compression volume. FLUIDPACK D-27 and KARNAVATI appear to be either backup machines, specialized equipment for specific formulations, or machines that experienced significant downtime during 2025.
 
**Recommendation:**
Verify whether FLUIDPACK D-27 and KARNAVATI are still active assets or candidates for decommissioning. If they are active, investigate the scheduling reason for their near-zero utilization — they represent untapped capacity that could relieve ZP-41 during Q3 peaks.

In [16]:
machine_coating_query = """
    SELECT
        m.machine_name,
        m.machine_type,
        COUNT(f.fact_id) AS batch_count
    FROM fact_batch_production f
    JOIN dim_machine m ON f.machine_id = m.machine_id
    WHERE f.machine_id IS NOT NULL
    AND f.stage = 'coating'
    GROUP BY m.machine_name, m.machine_type
    ORDER BY batch_count DESC;
"""

df_coating = pd.read_sql_query(machine_coating_query, engine)
df_coating

,machine_name,machine_type,batch_count
0,KEVIN 48,coater,441
1,KEVIN 66,coater,211
2,KEVIN 36,coater,35
3,ANCHORMARK,coater,4
4,KETO,coater,3
5,BGB 20,coater,2


In [17]:
fig = px.bar(
    df_coating,
    x='batch_count',
    y='machine_name',
    orientation='h',
    title='Machine Utilization — Coating (2025)',
    labels={'batch_count': 'Batch Count', 'machine_name': ''},
    text='batch_count',
    color='batch_count',
    color_continuous_scale='Blues'
)
fig.update_traces(textposition='auto', insidetextanchor='end')
fig.update_layout(
    plot_bgcolor='white',
    coloraxis_showscale=False,
    yaxis={'categoryorder': 'total ascending'},
    xaxis=dict(showgrid=False, title='')
)
fig.show()

## Coating
 
**Observation:**
KEVIN 48 handled 441 batches — more than double KEVIN 66's 211. KEVIN 36 (35), ANCHORMARK (4), KETO (3), and BGB 20 (2) are minimally used.
 
**Insight:**
The coating operation is heavily concentrated on two machines, with KEVIN 48 alone handling 63% of all coating batches. The four remaining coaters collectively handled only 44 batches, suggesting they are either backup equipment, specialized for specific coating types, or were offline for most of 2025.
 
**Recommendation:**
KEVIN 48 is a single point of failure risk for the coating operation. Any unplanned downtime on this machine would severely impact output, particularly for Film-Coated Tablets which represent 51% of total production volume. A contingency plan for KEVIN 48 downtime should be a priority.

In [18]:
machine_encapsulation_query = """
    SELECT
        m.machine_name,
        m.machine_type,
        COUNT(f.fact_id) AS batch_count
    FROM fact_batch_production f
    JOIN dim_machine m ON f.machine_id = m.machine_id
    WHERE f.machine_id IS NOT NULL
    AND f.stage = 'encapsulation'
    GROUP BY m.machine_name, m.machine_type
    ORDER BY batch_count DESC;
"""

df_encapsulation = pd.read_sql_query(machine_encapsulation_query, engine)
df_encapsulation

,machine_name,machine_type,batch_count
0,PHARMAFILL,encapsulation,94
1,NJP 3800,encapsulation,80


In [19]:
fig = px.bar(
    df_encapsulation,
    x='batch_count',
    y='machine_name',
    orientation='h',
    title='Machine Utilization — Encapsulation (2025)',
    labels={'batch_count': 'Batch Count', 'machine_name': ''},
    text='batch_count',
    color='batch_count',
    color_continuous_scale='Blues'
)
fig.update_traces(textposition='auto', insidetextanchor='end')
fig.update_layout(
    plot_bgcolor='white',
    coloraxis_showscale=False,
    yaxis={'categoryorder': 'total ascending'},
    xaxis=dict(showgrid=False, title='')
)
fig.show()

## Encapsulation
 
**Observation:**
PHARMAFILL handled 94 batches and NJP 3800 handled 80, for a total of 174 encapsulation batches — consistent with the capsule count in the dosage form breakdown.
 
**Insight:**
The two encapsulation machines have a relatively balanced split at 54/46, which is the healthiest utilization balance across all stages. Neither machine is overburdened and both are actively contributing.
 
**Recommendation:**
The encapsulation line is well-balanced. Maintain the current scheduling approach. As capsule demand grows, this stage has the most straightforward path to scaling — both machines are active and the workload is already evenly distributed.
 

# Module 3 — Machine Yield Analysis
## Average Yield by Machine vs Target (2025)
### Stages Covered: Compression | Coating | Encapsulation

In [20]:
machine_yield_compression_query = """
    SELECT
        m.machine_name,
        ROUND(AVG(f.yield_pct) * 100, 2) AS avg_yield_pct,
        COUNT(f.fact_id) AS batch_count
    FROM fact_batch_production f
    JOIN dim_machine m ON f.machine_id = m.machine_id
    WHERE f.stage = 'compression'
    AND f.yield_pct IS NOT NULL
    GROUP BY m.machine_name
    ORDER BY avg_yield_pct ASC;
"""

df_yield_compression = pd.read_sql_query(machine_yield_compression_query, engine)
df_yield_compression


,machine_name,avg_yield_pct,batch_count
0,KARNAVATI,87.72,2
1,FLUIDPACK D-27,92.41,15
2,ACCURA B,97.64,178
3,ZP-41,97.95,366
4,ACCURA D,97.96,293


In [29]:
fig = px.bar(
    df_yield_compression,
    x='avg_yield_pct',
    y='machine_name',
    orientation='h',
    title='Average Yield by Machine — Compression (2025)',
    labels={'avg_yield_pct': 'Average Yield (%)', 'machine_name': ''},
    text='avg_yield_pct',
    color='avg_yield_pct',
    color_continuous_scale='Blues'
)

fig.add_vline(
    x=97,
    line_dash='dash',
    line_color='red',
    annotation_text='Target: 97%',
    annotation_position='top right'
)

fig.update_traces(textposition='auto', insidetextanchor='end')
fig.update_layout(
    plot_bgcolor='white',
    coloraxis_showscale=False,
    yaxis={'categoryorder': 'total ascending'},
    xaxis=dict(showgrid=False, title='', range=[85, 100])
)
fig.show()


## Compression
**Target: 97%**
 
**Observation:**
ACCURA D (97.96%) and ZP-41 (97.95%) both meet the 97% target comfortably. ACCURA B (97.64%) is marginally above target. FLUIDPACK D-27 drops sharply to 92.41% and KARNAVATI is the worst performer at 87.72% — both well below the 97% target.
 
**Insight:**
The three high-volume machines are all performing at or above target, which is reassuring given they handle the bulk of compression batches. However FLUIDPACK D-27 and KARNAVATI's yield deficits are significant — 4.59 and 9.28 percentage points below target respectively. Crucially, these machines had only 15 and 2 batches — so their averages are statistically fragile and may reflect specific product-machine mismatches rather than chronic machine issues.
 
**Recommendation:**
Do not draw strong conclusions about FLUIDPACK D-27 and KARNAVATI from yield alone given their extremely low batch counts. Cross-reference which products were run on these machines — if they were assigned atypical formulations, the low yield may be a product issue rather than a machine issue. If scheduling allows, run a controlled comparison batch on these machines before flagging them for maintenance.

In [24]:
machine_yield_coating_query = """
    SELECT
        m.machine_name,
        ROUND(AVG(f.yield_pct) * 100, 2) AS avg_yield_pct,
        COUNT(f.fact_id) AS batch_count
    FROM fact_batch_production f
    JOIN dim_machine m ON f.machine_id = m.machine_id
    WHERE f.stage = 'coating'
    AND f.yield_pct IS NOT NULL
    GROUP BY m.machine_name
    ORDER BY avg_yield_pct ASC;
"""

df_yield_coating = pd.read_sql_query(machine_yield_coating_query, engine)
df_yield_coating

,machine_name,avg_yield_pct,batch_count
0,BGB 20,91.35,2
1,KEVIN 36,95.28,35
2,ANCHORMARK,95.46,4
3,KETO,96.21,3
4,KEVIN 48,96.80,441
5,KEVIN 66,97.55,211


In [26]:
fig = px.bar(
    df_yield_coating,
    x='avg_yield_pct',
    y='machine_name',
    orientation='h',
    title='Average Yield by Machine — Coating (2025)',
    labels={'avg_yield_pct': 'Average Yield (%)', 'machine_name': ''},
    text='avg_yield_pct',
    color='avg_yield_pct',
    color_continuous_scale='Blues'
)

fig.add_vline(
    x=96,
    line_dash='dash',
    line_color='red',
    annotation_text='Target: 96%',
    annotation_position='top right'
)

fig.update_traces(textposition='auto', insidetextanchor='end')
fig.update_layout(
    plot_bgcolor='white',
    coloraxis_showscale=False,
    yaxis={'categoryorder': 'total ascending'},
    xaxis=dict(showgrid=False, title='', range=[90, 100])
)
fig.show()

## Coating
**Target: 96%**
 
**Observation:**
KEVIN 66 leads at 97.55% and KEVIN 48 at 96.8% — both meeting the 96% target. KETO (96.21%) barely meets target. ANCHORMARK (95.46%) and KEVIN 36 (95.28%) fall below target. BGB 20 is the worst performer at 91.35%, nearly 5 points below target.
 
**Insight:**
The two machines carrying the highest workload — KEVIN 48 and KEVIN 66 — are both above target, which is the most operationally important finding. The underperforming machines (ANCHORMARK, KEVIN 36, BGB 20) all have very low batch counts (4, 35, and 2 respectively), which again raises the question of whether low yield reflects machine condition or product selection. BGB 20's 91.35% on just 2 batches is particularly unreliable as a performance indicator.
 
**Recommendation:**
Prioritize investigation of KEVIN 36 since it has a meaningful enough batch count (35) to warrant concern — a 95.28% average across 35 batches is a pattern, not noise. Review whether KEVIN 36 is receiving products with inherently difficult coating profiles. For BGB 20 and ANCHORMARK, withhold judgment until more batches are recorded.
 


In [30]:
machine_yield_encapsulation_query = """
    SELECT
        m.machine_name,
        ROUND(AVG(f.yield_pct) * 100, 2) AS avg_yield_pct,
        COUNT(f.fact_id) AS batch_count
    FROM fact_batch_production f
    JOIN dim_machine m ON f.machine_id = m.machine_id
    WHERE f.stage = 'encapsulation'
    AND f.yield_pct IS NOT NULL
    GROUP BY m.machine_name
    ORDER BY avg_yield_pct ASC;
"""

df_yield_encapsulation = pd.read_sql_query(machine_yield_encapsulation_query, engine)
df_yield_encapsulation

,machine_name,avg_yield_pct,batch_count
0,PHARMAFILL,95.44,94
1,NJP 3800,97.52,80


In [31]:
fig = px.bar(
    df_yield_encapsulation,
    x='avg_yield_pct',
    y='machine_name',
    orientation='h',
    title='Average Yield by Machine — Encapsulation (2025)',
    labels={'avg_yield_pct': 'Average Yield (%)', 'machine_name': ''},
    text='avg_yield_pct',
    color='avg_yield_pct',
    color_continuous_scale='Blues'
)

fig.add_vline(
    x=97,
    line_dash='dash',
    line_color='red',
    annotation_text='Target: 97%',
    annotation_position='top right'
)

fig.update_traces(textposition='auto', insidetextanchor='end')
fig.update_layout(
    plot_bgcolor='white',
    coloraxis_showscale=False,
    yaxis={'categoryorder': 'total ascending'},
    xaxis=dict(showgrid=False, title='', range=[94, 100])
)
fig.show()

## Encapsulation
**Target: 97%**
 
**Observation:**
NJP 3800 meets the 97% target at 97.52%. PHARMAFILL falls below at 95.44% — a 1.56 percentage point gap against the 97% target across 94 batches.
 
**Insight:**
Unlike the low-count outliers in compression and coating, PHARMAFILL's underperformance is statistically meaningful — 94 batches is a large enough sample to treat this as a real signal. A 95.44% average means roughly 1 in 22 batches on PHARMAFILL is losing material above the acceptable threshold. Given that PHARMAFILL handles the majority of encapsulation volume, this represents a consistent and quantifiable loss pattern.
 
**Recommendation:**
PHARMAFILL should be the highest priority for investigation across all three stages analyzed. With 94 batches below target, this is not a product issue — it is a machine issue. Conduct a maintenance review focusing on capsule fill weight consistency, tooling wear, and powder flow parameters. Quantify the material loss in absolute units and present it to management as a cost-recovery opportunity.
 

## Machine Frequency vs Yield Analysis
## Is There a Relationship Between Machine Usage and Yield Performance? (2025)
### Stages Covered: Compression | Coating | Encapsulation
 


In [32]:
machine_freq_yield_compression_query = """
    SELECT
        m.machine_name,
        COUNT(f.fact_id) AS batch_count,
        ROUND(AVG(f.yield_pct) * 100, 2) AS avg_yield_pct
    FROM fact_batch_production f
    JOIN dim_machine m ON f.machine_id = m.machine_id
    WHERE f.stage = 'compression'
    AND f.yield_pct IS NOT NULL
    GROUP BY m.machine_name
    ORDER BY batch_count DESC;
"""

df_freq_yield_compression = pd.read_sql_query(machine_freq_yield_compression_query, engine)
df_freq_yield_compression

,machine_name,batch_count,avg_yield_pct
0,ZP-41,366,97.95
1,ACCURA D,293,97.96
2,ACCURA B,178,97.64
3,FLUIDPACK D-27,15,92.41
4,KARNAVATI,2,87.72


In [33]:
fig = px.scatter(
    df_freq_yield_compression,
    x='batch_count',
    y='avg_yield_pct',
    text='machine_name',
    title='Machine Frequency vs Yield — Compression (2025)',
    labels={
        'batch_count': 'Batch Count (Frequency)',
        'avg_yield_pct': 'Average Yield (%)'
    },
    color='avg_yield_pct',
    color_continuous_scale='Blues',
    size='batch_count',
    size_max=40
)

fig.add_hline(
    y=97,
    line_dash='dash',
    line_color='red',
    annotation_text='Target: 97%',
    annotation_position='top right'
)

fig.update_traces(
    textposition='top center',
    marker=dict(line=dict(width=1, color='darkblue'))
)

fig.update_layout(
    plot_bgcolor='white',
    coloraxis_showscale=False,
    xaxis=dict(showgrid=True, gridcolor='lightgrey', title='Batch Count (Frequency)'),
    yaxis=dict(showgrid=True, gridcolor='lightgrey', title='Average Yield (%)', range=[85, 100])
)

fig.show()

## Compression
**Target: 97%**
 
**Observation:**
The three high-frequency machines — ZP-41 (366 batches), ACCURA D (293), and ACCURA B (178) — all cluster tightly above the 97% target line. The two low-frequency machines sit far below: FLUIDPACK D-27 at 15 batches and 92.41%, and KARNAVATI at just 2 batches and 87.72%. The pattern is visually clear — high usage machines are above the target line, low usage machines are below it.
 
**Insight:**
There is a strong positive relationship between machine frequency and yield in compression. However, the causal direction is ambiguous — it is not clear whether high-frequency machines perform better *because* they are used more (operator familiarity, consistent calibration, well-maintained workhorse status), or whether they are *assigned more batches because* they already perform well. FLUIDPACK D-27 and KARNAVATI's low batch counts make their yield averages statistically unreliable, but the gap is large enough to be worth noting.
 
**Recommendation:**
Before attributing the yield gap to machine condition, investigate what products were assigned to FLUIDPACK D-27 and KARNAVATI. If they were given difficult or atypical formulations, the low yield is a scheduling decision, not a maintenance problem. If the same products run at higher yields on ZP-41 or ACCURA D, then machine condition is the likely driver. A controlled product-machine comparison would resolve this.
 

In [34]:
machine_freq_yield_coating_query = """
    SELECT
        m.machine_name,
        COUNT(f.fact_id) AS batch_count,
        ROUND(AVG(f.yield_pct) * 100, 2) AS avg_yield_pct
    FROM fact_batch_production f
    JOIN dim_machine m ON f.machine_id = m.machine_id
    WHERE f.stage = 'coating'
    AND f.yield_pct IS NOT NULL
    GROUP BY m.machine_name
    ORDER BY batch_count DESC;
"""

df_freq_yield_coating = pd.read_sql_query(machine_freq_yield_coating_query, engine)
df_freq_yield_coating

,machine_name,batch_count,avg_yield_pct
0,KEVIN 48,441,96.80
1,KEVIN 66,211,97.55
2,KEVIN 36,35,95.28
3,ANCHORMARK,4,95.46
4,KETO,3,96.21
5,BGB 20,2,91.35


In [35]:
fig = px.scatter(
    df_freq_yield_coating,
    x='batch_count',
    y='avg_yield_pct',
    text='machine_name',
    title='Machine Frequency vs Yield — Coating (2025)',
    labels={
        'batch_count': 'Batch Count (Frequency)',
        'avg_yield_pct': 'Average Yield (%)'
    },
    color='avg_yield_pct',
    color_continuous_scale='Blues',
    size='batch_count',
    size_max=40
)

fig.add_hline(
    y=96,
    line_dash='dash',
    line_color='red',
    annotation_text='Target: 96%',
    annotation_position='top right'
)

fig.update_traces(
    textposition='top center',
    marker=dict(line=dict(width=1, color='darkblue'))
)

fig.update_layout(
    plot_bgcolor='white',
    coloraxis_showscale=False,
    xaxis=dict(showgrid=True, gridcolor='lightgrey', title='Batch Count (Frequency)'),
    yaxis=dict(showgrid=True, gridcolor='lightgrey', title='Average Yield (%)', range=[88, 100])
)

fig.show()

## Coating
**Target: 96%**
 
**Observation:**
The pattern in coating mirrors compression. KEVIN 48 (441 batches, 96.8%) and KEVIN 66 (211 batches, 97.55%) both sit above the 96% target. The low-frequency machines — KEVIN 36 (35 batches, 95.28%), ANCHORMARK (4 batches, 95.46%), KETO (3 batches, 96.21%), and BGB 20 (2 batches, 91.35%) — are clustered at the left edge of the scatter. BGB 20 is the lowest performer and also the least used.
 
**Insight:**
The same positive frequency-yield relationship holds in coating. High-usage machines meet or exceed target; low-usage machines tend to fall below it. KEVIN 36 is the most analytically meaningful data point among the underperformers — 35 batches is a large enough sample to treat its 95.28% average as a real signal rather than statistical noise. The remaining low-frequency machines (ANCHORMARK, KETO, BGB 20) have too few batches to draw firm conclusions.
 
**Recommendation:**
Focus investigative attention on KEVIN 36. With 35 batches below target, this machine warrants a formal yield review — examine whether it is consistently assigned to products with complex coating profiles (e.g. enteric-coated or modified release), and compare its batch-level yield distribution against KEVIN 48 running the same products. For BGB 20, ANCHORMARK, and KETO, accumulate more batches before drawing maintenance conclusions.


In [36]:
machine_freq_yield_encapsulation_query = """
    SELECT
        m.machine_name,
        COUNT(f.fact_id) AS batch_count,
        ROUND(AVG(f.yield_pct) * 100, 2) AS avg_yield_pct
    FROM fact_batch_production f
    JOIN dim_machine m ON f.machine_id = m.machine_id
    WHERE f.stage = 'encapsulation'
    AND f.yield_pct IS NOT NULL
    GROUP BY m.machine_name
    ORDER BY batch_count DESC;
"""

df_freq_yield_encapsulation = pd.read_sql_query(machine_freq_yield_encapsulation_query, engine)
df_freq_yield_encapsulation

,machine_name,batch_count,avg_yield_pct
0,PHARMAFILL,94,95.44
1,NJP 3800,80,97.52


In [37]:
fig = px.scatter(
    df_freq_yield_encapsulation,
    x='batch_count',
    y='avg_yield_pct',
    text='machine_name',
    title='Machine Frequency vs Yield — Encapsulation (2025)',
    labels={
        'batch_count': 'Batch Count (Frequency)',
        'avg_yield_pct': 'Average Yield (%)'
    },
    color='avg_yield_pct',
    color_continuous_scale='Blues',
    size='batch_count',
    size_max=40
)

fig.add_hline(
    y=97,
    line_dash='dash',
    line_color='red',
    annotation_text='Target: 97%',
    annotation_position='top right'
)

fig.update_traces(
    textposition='top center',
    marker=dict(line=dict(width=1, color='darkblue'))
)

fig.update_layout(
    plot_bgcolor='white',
    coloraxis_showscale=False,
    xaxis=dict(showgrid=True, gridcolor='lightgrey', title='Batch Count (Frequency)'),
    yaxis=dict(showgrid=True, gridcolor='lightgrey', title='Average Yield (%)', range=[93, 100])
)

fig.show()

## Encapsulation
**Target: 97%**
 
**Observation:**
Encapsulation presents the most striking finding across all three stages. NJP 3800, the lower-frequency machine at 80 batches, sits above the 97% target at 97.52%. PHARMAFILL, the higher-frequency machine at 94 batches, sits below target at 95.44%. This is the only stage where the relationship between frequency and yield is inverted — more usage corresponds to lower yield.
 
**Insight:**
Unlike compression and coating, encapsulation shows a negative frequency-yield relationship. The machine used more frequently — PHARMAFILL — is the one underperforming. This directly challenges the assumption that high-usage machines are better maintained or better calibrated. In PHARMAFILL's case, higher usage may be contributing to yield degradation, possibly through tooling wear, increased fill weight variability, or accumulated mechanical fatigue that has not been addressed through preventive maintenance.
 
**Recommendation:**
PHARMAFILL's underperformance is the highest-priority finding in this entire analysis. It is the most-used encapsulation machine, it operates below target, and the relationship between its frequency and yield runs in the wrong direction. Initiate a formal maintenance review of PHARMAFILL — specifically assess tooling condition, fill weight consistency across a sample of recent batches, and whether preventive maintenance intervals are being met. Quantify the yield gap in absolute unit terms and present it to management as a recoverable material loss. Consider temporarily redistributing batches to NJP 3800 while PHARMAFILL is assessed.